# Bound tightening: FBBT, row activity, and OBBT

Tightening variable bounds is one of the highest-leverage steps in global
optimization: a smaller box yields tighter convex relaxations, fewer branch-and-bound
nodes, and faster convergence {cite:p}`Puranik2017`. discopt runs a layered
bound-tightening stack:

1. **FBBT** (feasibility-based bound tightening) — propagate each constraint's *row
   activity* under the current bounds and isolate each variable, iterated across
   constraints to a fixpoint. Cheap (interval arithmetic, no LP) and exact on the
   constraint graph.
2. **Implied / row-activity propagation** — the linear specialization of the above.
3. **DBBT** (duality-based) — one objective LP's reduced costs tighten every variable
   at once when an incumbent cutoff is known.
4. **OBBT** (optimization-based) — solve ``min``/``max`` of each variable over the
   McCormick {cite:p}`McCormick1976` relaxation of the model, iterated as the box
   shrinks and the envelopes tighten {cite:p}`Tawarmalani2005`.

Every layer only ever derives *valid* bounds, so the tightened box always contains the
whole feasible region — bound tightening never removes the optimum.

This notebook focuses on the sound, LP-free core — FBBT — exposed publicly as
`discopt.tightening.fbbt_box`, and then shows the full stack at work inside a global
solve.


## FBBT row-activity propagation

Consider $a \ge b + 5$ and $b \ge c + 3$ with $a, b, c \in [0, 10]$. No single
constraint pins $a$, but propagating them together tightens the lower bounds: from the
second, $b \ge 3$; feeding that into the first, $a \ge 8$. FBBT reaches this fixpoint
automatically (and tightens the upper bounds the other way).


In [ ]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["JAX_ENABLE_X64"] = "1"

import numpy as np
import discopt.modeling.core as dm
from discopt.tightening import fbbt_box

m = dm.Model("chain")
a = m.continuous("a", lb=0, ub=10)
b = m.continuous("b", lb=0, ub=10)
c = m.continuous("c", lb=0, ub=10)
m.minimize(a + b + c)
m.subject_to(a - b >= 5)
m.subject_to(b - c >= 3)

res = fbbt_box(m)
print("tightened lower bounds:", np.round(res.lb, 3))   # -> [8, 3, 0]
print("tightened upper bounds:", np.round(res.ub, 3))   # -> [10, 5, 2]
print("variables tightened   :", res.n_tightened)


## Proving infeasibility

If propagation collapses a variable's interval to empty, the problem is infeasible —
a result that downstream tools (e.g. conflict analysis) use to emit no-good cuts.


In [ ]:
m2 = dm.Model("infeasible")
x = m2.continuous("x", lb=0, ub=10)
m2.minimize(x)
m2.subject_to(x >= 5)
m2.subject_to(x <= 3)

print("FBBT proves infeasible:", fbbt_box(m2).infeasible)


## The full stack inside a global solve

On a nonconvex MINLP the solver applies the whole stack — FBBT at presolve, then
DBBT and OBBT over the McCormick relaxation at the root — before and during spatial
branch-and-bound. The bounds only ever shrink, so the certified global optimum is
unchanged; the tightening just gets there with less search {cite:p}`Puranik2017`.


In [ ]:
m3 = dm.Model("nonconvex")
xi = m3.integer("xi", lb=0, ub=4)
y = m3.continuous("y", lb=0, ub=4)
m3.minimize((xi - 1.7) ** 2 + (y - 2.3) ** 2)
m3.subject_to(xi * y <= 3.0)   # nonconvex bilinear

result = m3.solve()
print(f"status: {result.status}, objective: {float(result.objective):.4f}")
